## Deprecated SDK imports removed

This project no longer depends on `bigdata-client` or `bigdata-research-tools`. See **MIGRATION_NOTES.md** / **README.md** and [Thematic_Screener_CLI](../Thematic_Screener_CLI/) for the REST + `bigdata-smart-batching` + OpenAI pattern. Pass company CSVs (`RP_ENTITY_ID`, `COMPANY_NAME`) instead of watchlists.


  # US Tariffs: Risks & Strategies - Report Generator

  ## Automated Analysis of Trade Tariff Risks and Corporate Mitigation Strategies

  ## Why It Matters







  In an era of increasing trade tensions and evolving geopolitical landscapes, companies face unprecedented uncertainty around import tariffs and trade barriers. Understanding corporate exposure to tariff risks across global supply chains is critical for investment decisions, risk management, and strategic planning. Manual tracking of tariff impacts across multiple companies and markets is time-intensive and often incomplete.

  ## What It Does



  This workflow combines an OpenAI-generated risk taxonomy, Bigdata.com REST + `bigdata-smart-batching` search, and the `GenerateReport` class to systematically analyze corporate exposure to US import tariff risks. Designed for portfolio managers, risk analysts, and trade compliance professionals, it transforms scattered information from news, filings, and earnings calls into a detailed research report covering risk intelligence and mitigation strategies.

  ## How It Works



  The workflow integrates **hybrid semantic search**, **AI-powered risk taxonomies**, and **multi-source content analysis** to deliver:



  - **Automated Risk Taxonomy Creation**: Uses OpenAI to generate hierarchical risk categories specific to tariff impacts

  - **Cross-Source Intelligence Gathering**: Searches news articles, SEC filings, and earnings transcripts for relevant discussions

  - **AI-Powered Risk Classification**: Categorizes content into specific risk scenarios

  - **Corporate Response Extraction**: Identifies and summarizes company mitigation plans from official communications

  - **Customizable Report Generation**: Produces professional HTML reports ranked by Media Attention, Financial Impact, and Uncertainty

  ## A Real-World Use Case







 This cookbook demonstrates the complete end-to-end workflow through analyzing how US import tariffs impact major American companies. You'll see how the system transforms scattered tariff discussions across news, SEC filings, and earnings transcripts into structured risk assessments, complete with corporate response strategies and quantified exposure metrics for investment and risk management decisions.

  ## Setup and Imports

  ## Async Compatibility Setup



  **Run this cell first** - Required for Google Colab, Jupyter Notebooks, and VS Code with Jupyter extension:



  ### Why is this needed?



  Interactive environments (Colab, Jupyter) already have an asyncio event loop running. Several helpers in this notebook's `src/` package (labeling, summarization, response extraction) make async calls to OpenAI, and without `nest_asyncio` you'll get this error:



  ```

  RuntimeError: asyncio.run() cannot be called from a running event loop

  ```



  The `nest_asyncio.apply()` command patches this to allow nested event loops.



  💡 **Tip**: If you're unsure which environment you're in, just run the cell below - it won't hurt in any environment!

In [1]:
import datetime
start = datetime.datetime.now()

try:
    import asyncio
    asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except (RuntimeError, ImportError):
    print("✅ nest_asyncio not needed or not available")

✅ nest_asyncio applied


  ## Environment Setup







  The following cell configures the necessary path for the analysis

In [2]:
import os
import sys


current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)
print(f"✅ Local environment setup complete")

✅ Local environment setup complete


  ## Optional: Plotly Display Configuration







  For better visualization rendering, you can also set the Plotly renderer:

In [3]:
import plotly.io as pio

# Try to detect the environment and set appropriate renderer
try:
    # Check if we're in JupyterLab
    import os
    if 'JUPYTERHUB_SERVICE_PREFIX' in os.environ or 'JPY_SESSION_NAME' in os.environ:
        pio.renderers.default = 'jupyterlab'
        print("✅ Plotly configured for JupyterLab")
    else:
        # Default for VS Code, Jupyter Notebook, etc.
        pio.renderers.default = 'plotly_mimetype+notebook'
        print("✅ Plotly configured for Jupyter/VS Code")
except:
    # Fallback to a more universal renderer
    pio.renderers.default = 'notebook'
    print("✅ Plotly configured with fallback renderer")

✅ Plotly configured for Jupyter/VS Code


  ## Configure Output Directories







  Set up the directory structure where analysis results and reports will be saved.

In [4]:
# Define output file paths for our report
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

  ## Load Credentials

In [5]:
from dotenv import load_dotenv
from pathlib import Path

script_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
load_dotenv(script_dir / '.env')

BIGDATA_API_KEY = os.getenv('BIGDATA_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not all([BIGDATA_API_KEY, OPENAI_API_KEY]):
    print("❌ Missing required environment variables")
    raise ValueError("Missing required environment variables. Check your .env file.")
else:
    print("✅ Credentials loaded from .env file")

✅ Credentials loaded from .env file


  ## Connecting to Bigdata







  Create a Bigdata object with your credentials.

In [6]:
# Bigdata.com access is now REST + bigdata-smart-batching, both authenticated
# directly with BIGDATA_API_KEY from the environment (loaded in the previous
# cell) -- there is no persistent SDK client object to construct anymore.
# See MIGRATION_PATTERNS.md / Thematic_Screener_CLI for the reference pattern.
os.environ.setdefault("BIGDATA_API_KEY", BIGDATA_API_KEY)
print("✅ Bigdata.com REST access ready (BIGDATA_API_KEY set)")


✅ Bigdata.com REST access ready (BIGDATA_API_KEY set)


  ## Import Required Libraries







  Import the core libraries needed for tariff risk analysis

In [7]:
from types import SimpleNamespace

from IPython.display import display, HTML
import pandas as pd

from src.bigdata_rest import load_universe, company_ids_from_universe
from src.mindmap.generate_trees import generate_themes_tree_dict, get_most_granular_elements
from src.mindmap.themes import print_tree
from src.search.content_retrieval import DataRetriever
from src.label.label_process import LabelProcessor
from src.report_generator import GenerateReport

print("✅ Core libraries imported (REST + bigdata-smart-batching + OpenAI pattern)")


✅ Core libraries imported (REST + bigdata-smart-batching + OpenAI pattern)


  ## Defining the Analysis Parameters



  - **Main Theme** (`main_theme`): The central risk scenario to analyze across companies

  - **Focus** (`focus`): Expert perspective for generating targeted risk taxonomies

  - **Company Universe** (`universe_df`): The set of companies to analyze, loaded from a CSV with `RP_ENTITY_ID` + `COMPANY_NAME` columns

  - **Model Selection** (`llm_model`): The AI model used for risk classification and summarization

  - **Time Period** (`start_date` and `end_date`): The date range for the analysis

  - **Frequency** (`freq`): The frequency of the date ranges to search over. Supported values:

     - `Y`: Yearly intervals.

     - `M`: Monthly intervals.

     - `W`: Weekly intervals.

     - `D`: Daily intervals. Defaults to `3M`.

  - **Document Limit** (`document_limit`): The maximum number of documents to return per query to Bigdata API.

  - **Batch Size** (`batch_size`): The number of entities to include in a single batched query.

  - **Rerank Threshold** (`rerank_threshold`): By setting this value, you’re enabling the cross-encoder which reranks the results and selects those whose relevance is above the percentile you specify (0.7 being the 70th percentile). More information on the re-ranker can be found [here](https://docs.bigdata.com/how-to-guides/rerank_search).

  - **Response From News** (`response_from_news`): Controls the `news_search_fallback` parameter. If `True`, when no response is found in transcripts/filings, the system uses News as fallback. In reports, fallback responses are annotated with `[From News]`. If `False`, missing responses show "No evidence of discussions found in Transcripts/Filings.". Default: `True`.



In [8]:
# ===== Customizable Parameters =====

from datetime import datetime, timedelta

# Company Universe: small slice (~5 companies) of the NASDAQ universe CSV
# bundled with Thematic_Screener_CLI (RP_ENTITY_ID + COMPANY_NAME), instead of
# a bigdata-client watchlist. Kept small to control API/LLM cost.
universe_path = "../Thematic_Screener_CLI/40_companies.csv"
universe_df = load_universe(universe_path).head(5).reset_index(drop=True)
company_ids = company_ids_from_universe(universe_df)
print(f"✅ Company universe loaded: {len(universe_df)} companies")
display(universe_df)

# Main Analysis Theme
main_theme = 'US Import Tariffs Corporate Risk Impact Analysis'
focus = "Provide a detailed taxonomy of risks describing how new American import tariffs will impact worldwide companies, their operations and strategy."

# LLM Model Configuration (plain OpenAI model id, passed straight to the OpenAI client)
llm_model = "gpt-4o-mini"

# Time Range Configuration -- kept to a ~30 day window to control API/LLM cost
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
freq = 'M'  # Monthly search frequency

# Enable/Disable Reranker
rerank_threshold = None

# Document Retrieval Limits (kept small to control OpenAI labeling/summary cost)
document_limit_news = 10
document_limit_filings = 5
batch_size = 1

# Toggle fallback to News for company responses
response_from_news = True


✅ Company universe loaded: 5 companies


,RP_ENTITY_ID,COMPANY_NAME
0,E09E2B,NVIDIA Corp.
1,D8442A,Apple Inc.
2,228D42,Microsoft Corp.
3,0157B1,Amazon.com Inc.
4,4A6F00,Alphabet Inc.


  ## Risk Analysis



  The first phase builds the risk taxonomy and retrieves/labels the News content that feeds the report generation phase (this replaces the deprecated `bigdata-research-tools` `RiskAnalyzer` class with local `src/` helpers built on REST + `bigdata-smart-batching` + OpenAI). This phase includes three critical steps that prepare the data for the report generation phase.

  ### Initialize Company Universe



  Sets up the company objects used for the risk discovery and taxonomy-driven search below:

  - **Automated Taxonomy Generation**: Creates a hierarchical structure of tariff-related risks

  - **Semantic Content Retrieval**: Searches news articles using the taxonomy's leaf summaries as queries

  - **Intelligent Content Labeling**: Categorizes found content into specific risk scenarios



In [9]:
# Build the company objects used for the News retrieval + labeling steps below.
# (GenerateReport builds these internally too, but we need the same objects
# here since retrieval/labeling for News happens before GenerateReport exists --
# this replaces RiskAnalyzer's internal entity resolution.)
id_to_name = dict(zip(universe_df["RP_ENTITY_ID"], universe_df["COMPANY_NAME"]))
list_entities = [SimpleNamespace(id=eid, name=name) for eid, name in id_to_name.items()]

print(f"✅ {len(list_entities)} companies ready for taxonomy-driven search: "
      f"{', '.join(e.name for e in list_entities)}")


✅ 5 companies ready for taxonomy-driven search: NVIDIA Corp., Apple Inc., Microsoft Corp., Amazon.com Inc., Alphabet Inc.


  ### Generate Risk Taxonomy







  Create a comprehensive taxonomy that breaks down tariff risks into specific, analyzable categories such as supply chain disruption, pricing impacts, and market access challenges.

In [10]:
# Generate a compact risk taxonomy for the theme/focus via OpenAI
# (replaces RiskAnalyzer.create_taxonomy())
themes_tree_dict = generate_themes_tree_dict(main_theme, focus)
risk_tree = themes_tree_dict[main_theme]
terminal_labels = get_most_granular_elements(risk_tree, 'Label')
risk_summaries = get_most_granular_elements(risk_tree, 'Summary')

print(f"✅ Taxonomy generated with {len(terminal_labels)} leaf risk categories")
print_tree(risk_tree)


2026-08-14 09:19:44,345 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Taxonomy generated with 6 leaf risk categories
US Import Tariffs Corporate Risk Impact Analysis
├── │   Operational Risks
│   ├── │   │   Supply Chain Disruptions
│   ├── │   │   Increased Costs
│   └── │       Compliance Challenges
└──     Strategic Risks
    ├──     │   Market Access Limitations
    ├──     │   Investment Decisions
    └──         Reputation Risks


  The taxonomy tree shows how tariff risks branch into specific sub-scenarios. Each terminal node represents a distinct risk category that will be used to classify and analyze news content.

  ### Retrieve Relevant Content







  Search news articles using the generated taxonomy to find discussions about tariff impacts across our company universe.

In [11]:
# Search news articles across the company universe using the taxonomy's leaf
# summaries as queries (replaces RiskAnalyzer.retrieve_results()).
data_retriever_news = DataRetriever(
    company_ids=company_ids,
    id_to_name=id_to_name,
    document_limit=document_limit_news,
    sortby="relevance",
    search_freq=freq,
    start_date_query=start_date,
    end_date_query=end_date,
)

df_sentences_semantic = data_retriever_news.retrieve(
    themes_tree_dict=themes_tree_dict,
    list_specific_themes=[main_theme],
    document_type="news",
)

if df_sentences_semantic is None:
    df_sentences_semantic = pd.DataFrame()

# Cost control: cap the number of chunks sent to OpenAI for labeling
df_sentences_semantic = df_sentences_semantic.head(document_limit_news)
print(f"✅ Retrieved {len(df_sentences_semantic)} news chunks (capped at {document_limit_news})")
df_sentences_semantic.head()


2026-08-14 09:19:44,373 - INFO - Planning search for text: 'Impact on sourcing materials and components due to increased costs or availability issues.'


2026-08-14 09:19:44,373 - INFO - Date range: 2026-07-15 to 2026-08-14


2026-08-14 09:19:44,374 - INFO - Using 1 entity IDs from inline list


2026-08-14 09:19:44,375 - INFO - Loaded 1 companies from universe


2026-08-14 09:19:44,377 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting


...

2026-08-14 09:20:55,530 - INFO - First pass complete: 14 documents with 14 chunks


2026-08-14 09:20:55,530 - INFO - Search complete: 14 documents with 14 chunks retrieved in 1.26s


2026-08-14 09:20:55,531 - INFO - Deduplicated: 14 unique documents from 14 total (chunks merged)


✅ Retrieved 10 news chunks (capped at 10)


,document_id,headline,timestamp,url,source_id,source_name,chunk_text,text,masked_text,relevance,sentiment,entity_id,entity_ids,entity_name,query,document_type,theme,entity_searched_id,entity_searched_name
0,6BD4C558F99E2E4B6D30005261AB1EE9,Americans are rallying against data centers. S...,2026-08-12T01:17:54,https://us.cnn.com/2026/08/06/business/ai-data...,2435A4,CNN,What's behind the delays?\nMaterials shortages...,What's behind the delays?\nMaterials shortages...,What's behind the delays?\nMaterials shortages...,0.185838,-0.60,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.
1,A4F0293E4168D93FEE46B691BF3BF73F,IBM share plunge is a warning to the IT sector,2026-07-15T07:57:15,https://www.ft.com/content/83aa00c5-e773-47be-...,DA9FC6,Financial Times,The other risk is that rising prices will cons...,The other risk is that rising prices will cons...,The other risk is that rising prices will cons...,0.126713,-0.28,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.
2,F3B0763201F7324906237FE21EE578E3,Transcript: The weaponisation of trade,2026-07-30T04:00:31,https://www.ft.com/content/9be25b80-1882-4107-...,DA9FC6,Financial Times,So I think this is a very powerful weapon for ...,So I think this is a very powerful weapon for ...,So I think this is a very powerful weapon for ...,0.110417,-0.35,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.
3,900688489DBDF8BC9BC983DA7A32A448,"""Look to Taiwan's supply chain for a peak-out ...",2026-08-04T21:07:31,https://magazine.hankyung.com/money/article/20...,2F93EE,Hankyung,"""The contract method has changed to LTA. While...","""The contract method has changed to LTA. While...","""The contract method has changed to LTA. While...",0.103011,-0.40,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.
4,B6CE2458767C79494D80035A9D18ED92,Oil prices sink nearly 7% and world shares gai...,2026-07-27T11:51:50,https://kstp.com/ap-top-news/ap-top-news-busin...,5E53F3,KSTP-TV,Higher energy costs are taking up a bigger sha...,Higher energy costs are taking up a bigger sha...,Higher energy costs are taking up a bigger sha...,0.107008,-0.42,E09E2B,[E09E2B],NVIDIA Corp.,Higher expenses related to tariffs leading to ...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.


  ### Labeling



  Use AI to analyze each news excerpt and categorize it into the appropriate risk scenarios. This creates structured data from unstructured news content.

In [12]:
# Classify each retrieved news excerpt into a risk category using OpenAI
# (replaces RiskAnalyzer.label_search_results()).
label_processor = LabelProcessor(
    list_entities=list_entities,
    themes_tree_dict=themes_tree_dict,
    list_specific_themes=[main_theme],
    api_key=OPENAI_API_KEY,
)

if df_sentences_semantic.empty:
    df_labeled = pd.DataFrame()
    print("⚠️ No news content retrieved for this window; skipping labeling.")
else:
    df_labeled = label_processor.run_label_process(df_sentences=df_sentences_semantic)
    if df_labeled is None:
        df_labeled = pd.DataFrame()
    print(f"✅ Labeled {len(df_labeled)} news excerpts")

df_labeled.head()


2026-08-14 09:20:56,732 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:56,787 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:56,796 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:56,804 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:56,825 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2026-08-14 09:20:56,999 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:57,353 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:20:57,355 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Completed 10 requests in 1.79 seconds.
Completed 0 requests in 0.00 seconds.
Completed 0 requests in 0.00 seconds.
Completed 0 requests in 0.00 seconds.
Completed 0 requests in 0.00 seconds.
✅ Labeled 10 news excerpts


,document_id,headline,timestamp,url,source_id,source_name,chunk_text,text,masked_text,relevance,...,entity_id,entity_ids,entity_name,query,document_type,theme,entity_searched_id,entity_searched_name,motivation,label
0,6BD4C558F99E2E4B6D30005261AB1EE9,Americans are rallying against data centers. S...,2026-08-12T01:17:54,https://us.cnn.com/2026/08/06/business/ai-data...,2435A4,CNN,What's behind the delays?\nMaterials shortages...,What's behind the delays?\nMaterials shortages...,What's behind the delays?\nMaterials shortages...,0.185838,...,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.,Target Company is not mentioned as being impac...,unclear
1,A4F0293E4168D93FEE46B691BF3BF73F,IBM share plunge is a warning to the IT sector,2026-07-15T07:57:15,https://www.ft.com/content/83aa00c5-e773-47be-...,DA9FC6,Financial Times,The other risk is that rising prices will cons...,The other risk is that rising prices will cons...,The other risk is that rising prices will cons...,0.126713,...,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.,Target Company does not explicitly mention how...,unclear
2,F3B0763201F7324906237FE21EE578E3,Transcript: The weaponisation of trade,2026-07-30T04:00:31,https://www.ft.com/content/9be25b80-1882-4107-...,DA9FC6,Financial Times,So I think this is a very powerful weapon for ...,So I think this is a very powerful weapon for ...,So I think this is a very powerful weapon for ...,0.110417,...,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.,Target Company does not explicitly mention any...,unclear
3,900688489DBDF8BC9BC983DA7A32A448,"""Look to Taiwan's supply chain for a peak-out ...",2026-08-04T21:07:31,https://magazine.hankyung.com/money/article/20...,2F93EE,Hankyung,"""The contract method has changed to LTA. While...","""The contract method has changed to LTA. While...","""The contract method has changed to LTA. While...",0.103011,...,E09E2B,[E09E2B],NVIDIA Corp.,Impact on sourcing materials and components du...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.,Target Company is discussing long-term contrac...,unclear
4,B6CE2458767C79494D80035A9D18ED92,Oil prices sink nearly 7% and world shares gai...,2026-07-27T11:51:50,https://kstp.com/ap-top-news/ap-top-news-busin...,5E53F3,KSTP-TV,Higher energy costs are taking up a bigger sha...,Higher energy costs are taking up a bigger sha...,Higher energy costs are taking up a bigger sha...,0.107008,...,E09E2B,[E09E2B],NVIDIA Corp.,Higher expenses related to tariffs leading to ...,news,US Import Tariffs Corporate Risk Impact Analysis,E09E2B,NVIDIA Corp.,Target Company is not discussed in the context...,unclear


  ## Report Generation



  The second phase uses `GenerateReport` and transforms the classified risk data into comprehensive reports with corporate mitigation strategies.

  ### Initialize GenerateReport







  The `GenerateReport` class will:



  - Create sector-wide risk summaries



  - Generate company-specific risk scores and summaries



  - Extract mitigation plans from SEC filings and earnings transcripts



  - Produce professional HTML reports with customizable ranking criteria

In [13]:
# Initialize the report generator with our analysis parameters
report_generator = GenerateReport(
        universe_df=universe_df,
        main_theme=main_theme,
        focus=focus,
        llm_model=llm_model,
        api_key=OPENAI_API_KEY,
        start_date=start_date,
        end_date=end_date,
        search_frequency=freq,
        document_limit_news=document_limit_news,
        document_limit_filings=document_limit_filings,
        batch_size=batch_size,
        themes_tree_dict=themes_tree_dict
)


  ### Generate Comprehensive Report







  Execute the complete report generation workflow including:



  1. **Sector-Level Summarization**: Create thematic summaries across risk categories



  2. **Company-Level Analysis**: Generate risk scores for Media Attention, Financial Impact, and Uncertainty



  3. **Mitigation Strategy Extraction**: Search filings and transcripts for corporate response plans (with News fallback when enabled via `news_search_fallback`)



  4. **Data Integration**: Combine all sources into structured report datasets

In [14]:
# Generate the risk report data
report = report_generator.generate_report(
    df_labeled=df_labeled,
    news_search_fallback = response_from_news, # Use response_from_news to enable/disable News fallback
    import_from_path=None,
    export_to_path=output_dir,
)

2026-08-14 09:20:59,451 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:21:01,036 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:21:01,115 - INFO - Exported summaries to pickle file.


2026-08-14 09:21:01,116 - INFO - Preparing topics from the labeled DataFrame


2026-08-14 09:21:01,119 - INFO - Starting processing for 2 tasks...


...

2026-08-14 09:23:14,738 - INFO - Starting process_response_by_company with 2 tasks...


2026-08-14 09:23:14,739 - INFO - Processing response summary for entity 'NVIDIA Corp.' and topic 'US Import Tariffs Corporate Risk Impact Analysis - Market Access Limitations'


2026-08-14 09:23:14,741 - INFO - Processing response summary for entity 'NVIDIA Corp.' and topic 'US Import Tariffs Corporate Risk Impact Analysis - Increased Costs'


2026-08-14 09:23:15,777 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-08-14 09:23:16,527 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  ## Final Output



  Transform the analysis results into professional, customizable reports. The system provides two distinct presentation styles, each optimized for different use cases and audiences.

  ### Report Customization Options







  Both report formats allow customization through multiple ranking criteria:







  **Sector-Wide Analysis**:



  - Identifies the most significant tariff risks across all companies



  - Ranks themes by media attention and document frequency



  - Provides executive summaries for each risk category







  **Company-Specific Analysis**:



  - **Most Reported Issue**: Highest media coverage and attention



  - **Biggest Risk**: Greatest potential financial impact



  - **Most Uncertain Issue**: Highest uncertainty scores and ambiguity







  Each company analysis includes extracted mitigation plans from official corporate communications, providing actionable intelligence for investment and risk management decisions.

  ### Report Format 1: Executive Summary Style







  This format prioritizes clarity and executive readability, focusing on the top risks per company across three key dimensions. Ideal for senior management briefings and board presentations.

In [15]:
from src.html_report import generate_html_report, prepare_data_report_0

# Extract report data for processing
df_by_theme = report.report_by_theme
df_by_company_with_responses = report.report_by_company

# Prepare data with executive summary formatting
top_by_theme, top_by_company = prepare_data_report_0(df_by_theme, df_by_company_with_responses)

# Generate executive-style HTML report
html_content = generate_html_report(top_by_theme, top_by_company, 'US Import Tariffs: Corporate Risk Impact Analysis')

# Save the executive report
report_filename = f'{output_dir}/tariffs_executive_report.html'
with open(report_filename, 'w') as file:
     file.write(html_content)

print(f"✅ Executive Report saved: {report_filename}")

✅ Executive Report saved: output/tariffs_executive_report.html


  #### Display Executive Report

In [16]:
display(HTML(html_content))

  ### Report Format 2: Detailed Analysis Version







  This format provides comprehensive risk analysis with extended company coverage and detailed risk breakdowns. Designed for analysts, portfolio managers, and risk management teams requiring in-depth insights.

In [17]:
from src.html_report import generate_html_report_v1, prepare_data_report_1

# Prepare data with detailed analysis formatting
top_by_theme, top_by_company = prepare_data_report_1(df_by_theme, df_by_company_with_responses)

# Generate detailed analysis HTML report
html_content_detailed = generate_html_report_v1(top_by_theme, top_by_company, 'US Import Tariffs: Comprehensive Risk Analysis')

# Save the detailed report
detailed_filename = f'{output_dir}/tariffs_detailed_analysis.html'
with open(detailed_filename, 'w') as file:
     file.write(html_content_detailed)

print(f"✅ Detailed Analysis Report saved: {detailed_filename}")

✅ Detailed Analysis Report saved: output/tariffs_detailed_analysis.html


  #### Display Detailed Analysis Report

In [18]:
display(HTML(html_content_detailed))

  ### Export Results for Further Analysis







  The generated data can be exported for integration with existing risk management systems, portfolio optimization tools, or compliance reporting workflows.

In [19]:
# Optional: Export structured data for external analysis
try:
    # Export the core datasets
    df_by_theme.to_csv(f'{output_dir}/tariffs_risks_by_theme.csv', index=False)
    df_by_company_with_responses.to_csv(f'{output_dir}/tariffs_risks_by_company.csv', index=False)
    
    print("✅ Data exported successfully:")
    print(f"   - Thematic analysis: {output_dir}/tariffs_risks_by_theme.csv")
    print(f"   - Company analysis: {output_dir}/tariffs_risks_by_company.csv")
    print(f"   - Executive report: {output_dir}/tariffs_executive_report.html")
    print(f"   - Detailed analysis: {output_dir}/tariffs_detailed_analysis.html")
    
except Exception as e:
    print(f"Warning: Export failed - {e}")

✅ Data exported successfully:
   - Thematic analysis: output/tariffs_risks_by_theme.csv
   - Company analysis: output/tariffs_risks_by_company.csv
   - Executive report: output/tariffs_executive_report.html
   - Detailed analysis: output/tariffs_detailed_analysis.html
